# Tool Calling

In [7]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [8]:
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    # Dictionary with some food items. This is static database for this example.
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }
    # It the food item is present in above created dictionary.
    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [9]:
get_food_calories('banana')

'Banana: 105 calories per medium banana (118g)'

In [10]:
get_food_calories('grapes')

"I don't have calorie data for grapes in my database. Try common foods like apple, chicken breast, or rice."

Copying the above function to make it into a function_tool. With this markdown/tag one can convert a simple agentic function into an OpenAI Function Tool.

In [ ]:
# after adding function_tool to this function, the get_food_calories('') will not work
@function_tool
def get_food_calories(food_item: str) -> str:
    # LLM and OpenAI uses this below Doc string to figure out what this function does. The Agent/LLM would not know how the written code works
    # The LLM is fully dependent on the Doc-string to understand what a certain function in a tool does.
    # This makes the doc-string important part of building an AI Agent. You need to provide a verbose docstring.
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    # Dictionary with some food items. This is static database for this example.
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }
    # It the food item is present in above created dictionary.
    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

In [12]:
get_food_calories('grapes')

TypeError: 'FunctionTool' object is not callable

But now I can create an agent and use the above created function_tool in this agent.

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    # In the instructions above, we can add 'Use the get_food_calories for retrieving calorie information' and thus define which tool the
    # agent should use for calorie retrieval.
    # adding the tools line to specify which tool to use
    tools=[get_food_calories]
    )

In [14]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 185 calories total (banana ~105 + apple ~80 for medium sizes).


The agent did two tool calls for the above function; once for the banana and once for apple. Refer the doc for more infromation. If the agent still does not use the tool, you can enfore the agent to use the tool as follows:

Enforce tools use:

In [15]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 185 calories total (banana ~105 cal + apple ~80 cal for medium sizes; actual totals vary with size).
